# Step 1: Load Source Data
Load the two raw CSV files and confirm the dataset shape before any transformations.

In [29]:
raw_data_path <- "data/raw/diabetic_data.csv"
ids_map_path <- "data/raw/IDS_mapping.csv"

diabetic <- read.csv(
  raw_data_path,
  stringsAsFactors = FALSE,
  na.strings = c("", "?", "NULL")
)

ids_raw <- read.csv(
  ids_map_path,
  header = FALSE,
  stringsAsFactors = FALSE,
  fill = TRUE,
  na.strings = c("", "NULL")
)

colnames(ids_raw) <- c("key", "description")

cat("diabetic_data.csv shape:", sprintf("(%d, %d)", nrow(diabetic), ncol(diabetic)), "\n")
cat("IDS_mapping raw shape:", sprintf("(%d, %d)", nrow(ids_raw), ncol(ids_raw)), "\n\n")

old_width <- getOption("width")
options(width = 1000)

cat("diabetic_data preview (all columns):\n")
print(head(diabetic))

options(width = old_width)



diabetic_data.csv shape: (101766, 50) 
IDS_mapping raw shape: (68, 2) 

diabetic_data preview (all columns):
  encounter_id patient_nbr            race gender     age weight admission_type_id discharge_disposition_id admission_source_id time_in_hospital payer_code        medical_specialty num_lab_procedures num_procedures num_medications number_outpatient number_emergency number_inpatient diag_1 diag_2 diag_3 number_diagnoses max_glu_serum A1Cresult metformin repaglinide nateglinide chlorpropamide glimepiride acetohexamide glipizide glyburide tolbutamide pioglitazone rosiglitazone acarbose miglitol troglitazone tolazamide examide citoglipton insulin glyburide.metformin glipizide.metformin glimepiride.pioglitazone metformin.rosiglitazone metformin.pioglitazone change diabetesMed readmitted
1      2278392     8222157       Caucasian Female  [0-10)   <NA>                 6                       25                   1                1       <NA> Pediatrics-Endocrinology                 41 

# Step 2: Parse IDS Mapping Tables
Split IDS_mapping.csv into three mapping tables: admission type, discharge disposition, and admission source.

In [30]:
header_rows <- which(grepl("_id$", ids_raw$key) & ids_raw$description == "description")

extract_mapping <- function(section_name) {
  start_idx <- which(ids_raw$key == section_name & ids_raw$description == "description")
  if (length(start_idx) == 0) return(data.frame(id = integer(), description = character()))
  start_idx <- start_idx[1]

  next_headers <- header_rows[header_rows > start_idx]
  end_idx <- if (length(next_headers) > 0) next_headers[1] - 1 else nrow(ids_raw)

  out <- ids_raw[(start_idx + 1):end_idx, , drop = FALSE]
  out <- out[!is.na(out$key) & nzchar(trimws(out$key)), , drop = FALSE]
  out$id <- suppressWarnings(as.integer(out$key))
  out <- out[!is.na(out$id), c("id", "description"), drop = FALSE]
  rownames(out) <- NULL
  out
}

admission_type_map <- extract_mapping("admission_type_id")
discharge_disposition_map <- extract_mapping("discharge_disposition_id")
admission_source_map <- extract_mapping("admission_source_id")

cat("admission_type_map preview:\n")
print(head(admission_type_map, 5))

cat("\ndischarge_disposition_map preview:\n")
print(head(discharge_disposition_map, 5))

cat("\nadmission_source_map preview:\n")
print(head(admission_source_map, 5))

admission_type_map preview:
  id   description
1  1     Emergency
2  2        Urgent
3  3      Elective
4  4       Newborn
5  5 Not Available

discharge_disposition_map preview:
  id                                                          description
1  1                                                   Discharged to home
2  2                Discharged/transferred to another short term hospital
3  3                                        Discharged/transferred to SNF
4  4                                        Discharged/transferred to ICF
5  5 Discharged/transferred to another type of inpatient care institution

admission_source_map preview:
  id                                     description
1  1                              Physician Referral
2  2                                 Clinic Referral
3  3                                    HMO Referral
4  4                        Transfer from a hospital
5  5  Transfer from a Skilled Nursing Facility (SNF)


# Step 3: Data Cleaning

In [ ]:
# Patient-level deduplication: Keep only the first encounter for each patient_nbr
rows_before <- nrow(diabetic)

encounter_num <- suppressWarnings(as.numeric(diabetic$encounter_id))
order_idx <- order(diabetic$patient_nbr, encounter_num, na.last = TRUE)
diabetic <- diabetic[order_idx, , drop = FALSE]
diabetic <- diabetic[!duplicated(diabetic$patient_nbr), , drop = FALSE]

rows_after <- nrow(diabetic)
cat("Patient-level deduplication complete (first encounter per patient_nbr kept).\n")
cat("Rows before:", rows_before, "| Rows after:", rows_after, "\n")

Patient-level deduplication complete (first encounter per patient_nbr kept).
Rows before: 101766 | Rows after: 71518 


In [ ]:
# Drop columns with high missingness or low relevance
columns_to_drop <- c("weight", "medical_specialty", "payer_code")

dropped_columns <- c()
for (col_name in columns_to_drop) {
  if (col_name %in% names(diabetic)) {
    diabetic <- diabetic[, names(diabetic) != col_name, drop = FALSE]
    dropped_columns <- c(dropped_columns, col_name)
  }
}

if (length(dropped_columns) > 0) {
  cat("Dropped columns:", paste(dropped_columns, collapse = ", "), "\n")
} else {
  cat("No target columns were present to drop.\n")
}

already_absent <- setdiff(columns_to_drop, dropped_columns)
if (length(already_absent) > 0) {
  cat("Already absent:", paste(already_absent, collapse = ", "), "\n")
}

cat("Updated diabetic_data shape:", sprintf("(%d, %d)", nrow(diabetic), ncol(diabetic)), "\n")

Dropped columns: weight, medical_specialty, payer_code 
Updated diabetic_data shape: (71518, 47) 


In [ ]:
# Impute missing
if ("race" %in% names(diabetic)) {
  na_before <- sum(is.na(diabetic$race))
  diabetic$race[is.na(diabetic$race)] <- "Other"
  na_after <- sum(is.na(diabetic$race))

  cat("Filled", na_before, "missing values in race with 'Other'.\n")
  cat("Remaining missing values in race:", na_after, "\n")
} else {
  cat("Column 'race' not found; no imputation applied.\n")
}

for (lab_col in c("max_glu_serum", "A1Cresult")) {
  if (lab_col %in% names(diabetic)) {
    na_before <- sum(is.na(diabetic[[lab_col]]))
    diabetic[[lab_col]][is.na(diabetic[[lab_col]])] <- "None"
    cat("Filled", na_before, "missing values in", lab_col, "with 'None'.\n")
  }
}

Filled 1948 missing values in race with 'Other'.
Remaining missing values in race: 0 
Filled 0 missing values in max_glu_serum with 'None'.
Filled 0 missing values in A1Cresult with 'None'.


In [ ]:
# Drop rows with invalid or missing critical values
if ("gender" %in% names(diabetic)) {
  invalid_mask <- trimws(as.character(diabetic$gender)) == "Unknown/Invalid"
  invalid_mask[is.na(invalid_mask)] <- FALSE

  invalid_count <- sum(invalid_mask)
  diabetic <- diabetic[!invalid_mask, , drop = FALSE]

  cat("Dropped", invalid_count, "rows where gender == 'Unknown/Invalid'.\n")
} else {
  cat("Column 'gender' not found; no rows dropped.\n")
}

if ("diag_1" %in% names(diabetic)) {
  diag1_missing <- is.na(diabetic$diag_1) | trimws(as.character(diabetic$diag_1)) %in% c("", "?")
  diag1_drop_count <- sum(diag1_missing)
  diabetic <- diabetic[!diag1_missing, , drop = FALSE]
  cat("Dropped", diag1_drop_count, "rows with missing diag_1.\n")
} else {
  cat("Column 'diag_1' not found; no rows dropped for primary diagnosis.\n")
}

cat("Updated row count after row-level cleaning:", nrow(diabetic), "\n")

Dropped 3 rows where gender == 'Unknown/Invalid'.
Dropped 11 rows with missing diag_1.
Updated row count after row-level cleaning: 71504 


In [ ]:
# Map diagnosis codes to categories
map_diag_category <- function(code, allow_none = FALSE) {
  code_chr <- toupper(trimws(as.character(code)))

  if (is.na(code) || code_chr == "" || code_chr == "?") {
    return(if (allow_none) "None" else NA_character_)
  }

  if (grepl("^[VE]", code_chr)) {
    return("Other")
  }

  code_num <- suppressWarnings(as.numeric(code_chr))
  if (is.na(code_num)) {
    return("Other")
  }

  if ((code_num >= 390 && code_num <= 459) || code_num == 785) return("Circulatory")
  if ((code_num >= 460 && code_num <= 519) || code_num == 786) return("Respiratory")
  if ((code_num >= 520 && code_num <= 579) || code_num == 787) return("Digestive")
  if (code_num >= 250 && code_num < 251) return("Diabetes")
  if (code_num >= 800 && code_num <= 999) return("Injury")
  if (code_num >= 710 && code_num <= 739) return("Musculoskeletal")
  if ((code_num >= 580 && code_num <= 629) || code_num == 788) return("Genitourinary")
  if (code_num >= 140 && code_num <= 239) return("Neoplasms")

  return("Other")
}

diabetic$diag_1 <- vapply(diabetic$diag_1, function(x) map_diag_category(x, allow_none = FALSE), character(1))
diabetic$diag_2 <- vapply(diabetic$diag_2, function(x) map_diag_category(x, allow_none = TRUE), character(1))
diabetic$diag_3 <- vapply(diabetic$diag_3, function(x) map_diag_category(x, allow_none = TRUE), character(1))

diag1_post_map_missing <- is.na(diabetic$diag_1)
if (any(diag1_post_map_missing)) {
  diabetic <- diabetic[!diag1_post_map_missing, , drop = FALSE]
  cat("Dropped", sum(diag1_post_map_missing), "additional rows with missing diag_1 after mapping.\n")
}

cat("diag_1 category counts (top 10):\n")
print(head(sort(table(diabetic$diag_1), decreasing = TRUE), 10))

cat("\ndiag_2 category counts (top 10):\n")
print(head(sort(table(diabetic$diag_2), decreasing = TRUE), 10))

cat("\ndiag_3 category counts (top 10):\n")
print(head(sort(table(diabetic$diag_3), decreasing = TRUE), 10))

diag_1 category counts (top 10):

    Circulatory           Other     Respiratory       Digestive        Diabetes 
          21893           12347            9776            6570            5805 
         Injury Musculoskeletal   Genitourinary       Neoplasms 
           4777            4080            3514            2742 

diag_2 category counts (top 10):

    Circulatory           Other        Diabetes     Respiratory   Genitourinary 
          22532           18404            9756            7242            5467 
      Digestive          Injury       Neoplasms Musculoskeletal            None 
           2907            1856            1750            1297             293 

diag_3 category counts (top 10):

    Circulatory           Other        Diabetes     Respiratory   Genitourinary 
          21308           20416           12659            4872            4198 
      Digestive          Injury Musculoskeletal       Neoplasms            None 
           2746            1441      

In [ ]:
# Map ID columns to descriptive labels
map_id_to_label <- function(df, id_col, mapping_df) {
  if (!(id_col %in% names(df))) return(df[[id_col]])
  lookup <- setNames(mapping_df$description, as.character(mapping_df$id))
  out <- lookup[as.character(df[[id_col]])]
  out[is.na(out)] <- "Unknown"
  unname(out)
}

if (exists("admission_type_map") && exists("discharge_disposition_map") && exists("admission_source_map")) {
  diabetic$admission_type_id <- map_id_to_label(diabetic, "admission_type_id", admission_type_map)
  diabetic$discharge_disposition_id <- map_id_to_label(diabetic, "discharge_disposition_id", discharge_disposition_map)
  diabetic$admission_source_id <- map_id_to_label(diabetic, "admission_source_id", admission_source_map)

  cat("Mapped admission/discharge/source ID columns to descriptive labels.\n")
} else {
  cat("Mapping tables not found; run Step 2 first.\n")
}

Mapped admission/discharge/source ID columns to descriptive labels.


In [ ]:
# Create binary target variable for readmission within 30 days
if ("readmitted" %in% names(diabetic)) {
  diabetic$readmitted_binary <- ifelse(diabetic$readmitted == "<30", 1L, 0L)
  cat("Created readmitted_binary: 1 for <30, 0 for >30 or NO.\n")
  cat("Class balance (readmitted_binary):\n")
  print(table(diabetic$readmitted_binary))
} else {
  cat("Column 'readmitted' not found; binary target not created.\n")
}

Created readmitted_binary: 1 for <30, 0 for >30 or NO.
Class balance (readmitted_binary):

    0     1 
65213  6291 
